# V1-S11 — Dual-novelty 2×2 archetype, temporal trends, Gate G3 inputs

This notebook **loads** the V1-S11 archetypes artifact (`data/v1/archetypes.parquet`) and computes the numbers that **Gate G3** consumes — the semantic×structural correlation table, the 2×2 archetype distributions, temporal / journal / impact breakdowns, and the F2 small-multiple figure.

It performs **no network calls** and does **NOT recompute the binning** — the four `arch_*` label columns and the corpus-frozen median thresholds were produced upstream by `scifield novelty archetypes`. We read the thresholds straight from the parquet sidecar so the median crosshairs in F2 match the exact cut points used for binning.

The semantic axis is always listed **first**, the structural axis **second**. The four pairings are `sem_nov_mean×cd5`, `sem_nov_mean×cd10`, `sem_nov_min×cd5`, `sem_nov_min×cd10`. **mean×cd5** is the primary pairing for temporal / journal / impact views.

## 1. Setup / load

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Repo-root sniff — notebook runs from notebooks/, code lives one dir up.
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

DATA = repo_root / "data" / "v1"
FIGURES_DIR = repo_root / "docs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

from scifield.novelty.archetypes import QUADRANT_LABELS  # noqa: E402
from scifield.repro import record_run  # noqa: E402

ARCH_PATH = DATA / "archetypes.parquet"
SIDECAR_PATH = Path(str(ARCH_PATH) + ".run.json")
DPI = 120

In [2]:
# Load the archetypes parquet + its provenance sidecar. Read the corpus-frozen
# median thresholds FROM the sidecar — never recompute them here.
df = pd.read_parquet(ARCH_PATH)
sidecar = json.loads(SIDECAR_PATH.read_text())
thresholds = sidecar["config"]["thresholds"]

n_total = len(df)
print(f"archetypes parquet     : {ARCH_PATH}")
print(f"columns                : {list(df.columns)}")
print(f"n_total (all rows)     : {n_total:,}")
print(f"sidecar n_total        : {sidecar['config']['n_total']:,}")
print(f"sidecar n_complete     : {sidecar['config']['n_complete_cases']:,}")
print("frozen median thresholds (from sidecar):")
for k, v in thresholds.items():
    print(f"    {k:>14} = {v:.10f}")

archetypes parquet     : /Users/samersalman/Desktop/SciField/data/v1/archetypes.parquet
columns                : ['pmid', 'year', 'journal', 'topic_id', 'sem_nov_mean', 'sem_nov_min', 'cd5', 'cd10', 'cited_by_count', 'cited_by_pctile_within_year', 'arch_mean_cd5', 'arch_mean_cd10', 'arch_min_cd5', 'arch_min_cd10', 'n_prior', 'openalex_id']
n_total (all rows)     : 89,230
sidecar n_total        : 89,230
sidecar n_complete     : 81,733
frozen median thresholds (from sidecar):
              cd10 = -0.4545454545
               cd5 = -0.5000000000
      sem_nov_mean = 0.4348148205
       sem_nov_min = 0.1461571455


In [3]:
# Complete-case subset = all four metric axes simultaneously non-null. The four
# correlations and the 2x2 binning all share this exact set of rows.
METRICS = ["sem_nov_mean", "sem_nov_min", "cd5", "cd10"]
COMPLETE = df.dropna(subset=METRICS).copy()
print(f"complete cases (all 4 non-null): {len(COMPLETE):,}")
assert len(COMPLETE) == 81_733, len(COMPLETE)
print("complete-case assertion PASS — 81,733")

# The four pairings (semantic first, structural second) + their arch label cols.
PAIRINGS = [
    ("sem_nov_mean", "cd5"),
    ("sem_nov_mean", "cd10"),
    ("sem_nov_min", "cd5"),
    ("sem_nov_min", "cd10"),
]


def arch_col(sem: str, struct: str) -> str:
    """Map a pairing to its emitted arch_* column name (sem_nov_ stripped)."""
    return f"arch_{sem.replace('sem_nov_', '')}_{struct}"


for sem, struct in PAIRINGS:
    col = arch_col(sem, struct)
    assert col in df.columns, col
print("pairings ->", [(s, t, arch_col(s, t)) for s, t in PAIRINGS])

complete cases (all 4 non-null): 81,733
complete-case assertion PASS — 81,733
pairings -> [('sem_nov_mean', 'cd5', 'arch_mean_cd5'), ('sem_nov_mean', 'cd10', 'arch_mean_cd10'), ('sem_nov_min', 'cd5', 'arch_min_cd5'), ('sem_nov_min', 'cd10', 'arch_min_cd10')]


## 2. Correlation table — THE Gate G3 numbers

For each of the four pairings, Pearson `r` and Spearman `ρ` between the semantic metric (x) and the structural CD metric (y), on the **complete-case set** (N=81,733 for all four). The gate's row-1 numbers are `max_abs_pearson` and `max_abs_spearman` — the largest `|r|` and `|ρ|` across the four pairings (most conservative). A *low* dual-novelty correlation is the evidence that semantic and structural novelty are **distinct** axes worth crossing into a 2×2.

In [4]:
try:
    from scipy.stats import pearsonr, spearmanr

    def _pearson(x, y):
        return float(pearsonr(x, y)[0])

    def _spearman(x, y):
        return float(spearmanr(x, y)[0])

    CORR_BACKEND = "scipy.stats"
except Exception:  # pragma: no cover - scipy is expected to be present

    def _pearson(x, y):
        return float(pd.Series(x).corr(pd.Series(y), method="pearson"))

    def _spearman(x, y):
        return float(pd.Series(x).corr(pd.Series(y), method="spearman"))

    CORR_BACKEND = "pandas.corr"
print(f"correlation backend: {CORR_BACKEND}")

corr_rows = []
for sem, struct in PAIRINGS:
    x = COMPLETE[sem].to_numpy()
    y = COMPLETE[struct].to_numpy()
    corr_rows.append(
        {
            "pairing": f"{sem} × {struct}",
            "n": int(len(COMPLETE)),
            "pearson_r": _pearson(x, y),
            "spearman_rho": _spearman(x, y),
        }
    )
corr_df = pd.DataFrame(corr_rows)
with pd.option_context("display.float_format", lambda v: f"{v:.6f}"):
    display(corr_df)

correlation backend: scipy.stats


,pairing,n,pearson_r,spearman_rho
0,sem_nov_mean × cd5,81733,0.077938,0.080488
1,sem_nov_mean × cd10,81733,0.082225,0.090566
2,sem_nov_min × cd5,81733,0.166297,0.145775
3,sem_nov_min × cd10,81733,0.182812,0.175121


In [5]:
# Load-bearing gate numbers: the largest |r| and |rho| across the four pairings.
ip = corr_df["pearson_r"].abs().idxmax()
is_ = corr_df["spearman_rho"].abs().idxmax()
max_abs_pearson = float(abs(corr_df.loc[ip, "pearson_r"]))
max_abs_spearman = float(abs(corr_df.loc[is_, "spearman_rho"]))
max_abs_pearson_pairing = corr_df.loc[ip, "pairing"]
max_abs_spearman_pairing = corr_df.loc[is_, "pairing"]

pearson_ok = max_abs_pearson < 0.4
spearman_ok = max_abs_spearman < 0.4

print("=" * 64)
print("GATE G3 row-1 numbers — dual-novelty correlation")
print("=" * 64)
print(f"  max_abs_pearson  = {max_abs_pearson:.6f}  ({max_abs_pearson_pairing})")
print(f"  max_abs_spearman = {max_abs_spearman:.6f}  ({max_abs_spearman_pairing})")
print("-" * 64)
print(f"  max_abs_pearson  < 0.4 -> {pearson_ok}")
print(f"  max_abs_spearman < 0.4 -> {spearman_ok}")
print("=" * 64)

GATE G3 row-1 numbers — dual-novelty correlation
  max_abs_pearson  = 0.182812  (sem_nov_min × cd10)
  max_abs_spearman = 0.175121  (sem_nov_min × cd10)
----------------------------------------------------------------
  max_abs_pearson  < 0.4 -> True
  max_abs_spearman < 0.4 -> True


## 3. F2 figure — 2×2 small-multiple panel

One subplot per pairing: semantic metric (x) vs structural CD metric (y) over the 81,733 complete cases, as a `hexbin` density (keeps the PNG well under 1 MB and stays legible at ~81k points). Dashed median crosshairs = the corpus-frozen thresholds; subplot title carries the pairing's Pearson r and Spearman ρ; quadrant corners annotated with the archetype names.

In [6]:
F2_FIG = FIGURES_DIR / "F2_dual_novelty.png"
GRIDSIZE = 60

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, (sem, struct) in zip(axes.ravel(), PAIRINGS, strict=False):
    x = COMPLETE[sem].to_numpy()
    y = COMPLETE[struct].to_numpy()
    hb = ax.hexbin(x, y, gridsize=GRIDSIZE, cmap="viridis", mincnt=1)
    fig.colorbar(hb, ax=ax, label="count", shrink=0.85)

    tx = thresholds[sem]
    ty = thresholds[struct]
    ax.axvline(tx, color="red", ls="--", lw=1.0)
    ax.axhline(ty, color="red", ls="--", lw=1.0)

    row = corr_df[corr_df["pairing"] == f"{sem} × {struct}"].iloc[0]
    r = row["pearson_r"]
    rho = row["spearman_rho"]
    ax.set_title(f"{sem} × {struct}\nr={r:.3f}, ρ={rho:.3f}  (N={int(row['n']):,})")
    ax.set_xlabel(f"{sem} (semantic novelty)")
    ax.set_ylabel(f"{struct} (structural disruption)")

    # Annotate the four quadrant corners with the archetype names.
    xlo, xhi = ax.get_xlim()
    ylo, yhi = ax.get_ylim()
    corner_kw = dict(fontsize=8, color="white", fontweight="bold", alpha=0.9)
    # HH = High sem (right) + High struct (top); LL = Low/Low (bottom-left); etc.
    ax.text(xhi, yhi, QUADRANT_LABELS["HH"], ha="right", va="top", **corner_kw)
    ax.text(xlo, yhi, QUADRANT_LABELS["LH"], ha="left", va="top", **corner_kw)
    ax.text(xhi, ylo, QUADRANT_LABELS["HL"], ha="right", va="bottom", **corner_kw)
    ax.text(xlo, ylo, QUADRANT_LABELS["LL"], ha="left", va="bottom", **corner_kw)

fig.suptitle("F2 — Dual-novelty archetype: semantic × structural", fontsize=14)
fig.tight_layout()
fig.savefig(F2_FIG, dpi=DPI)
plt.close(fig)
print("wrote", F2_FIG)

wrote /Users/samersalman/Desktop/SciField/docs/figures/F2_dual_novelty.png


In [7]:
# Enforce the <1 MB figure budget; shrink dpi if needed and re-save.
sz = F2_FIG.stat().st_size
print(f"F2 size = {sz / 1024:.1f} KB  (dpi={DPI}, hexbin gridsize={GRIDSIZE})")
assert sz < 1_000_000, f"F2 too large: {sz} bytes"
print("F2 size assertion PASS — under 1 MB")

F2 size = 510.1 KB  (dpi=120, hexbin gridsize=60)
F2 size assertion PASS — under 1 MB


## 4. Distribution table — per-quadrant counts + shares

Per-pairing counts and shares over the archetype label column (nulls dropped). Shares are within the binned (complete-on-that-pairing) population.

In [8]:
dist_rows = []
for sem, struct in PAIRINGS:
    col = arch_col(sem, struct)
    vc = df[col].value_counts(dropna=True)
    total = int(vc.sum())
    for label in QUADRANT_LABELS.values():
        n = int(vc.get(label, 0))
        dist_rows.append(
            {
                "pairing": f"{sem} × {struct}",
                "archetype": label,
                "n": n,
                "share": n / total if total else float("nan"),
            }
        )
dist_df = pd.DataFrame(dist_rows)
with pd.option_context("display.float_format", lambda v: f"{v:.4f}"):
    display(dist_df)

print("\nmean×cd5 (primary) per-quadrant counts + shares:")
primary_dist = dist_df[dist_df["pairing"] == "sem_nov_mean × cd5"].reset_index(drop=True)
with pd.option_context("display.float_format", lambda v: f"{v:.4f}"):
    display(primary_dist)

,pairing,archetype,n,share
0,sem_nov_mean × cd5,disruptive-novel,22662,0.2773
1,sem_nov_mean × cd5,novel-consolidating,18210,0.2228
2,sem_nov_mean × cd5,conventional-disruptive,19474,0.2382
3,sem_nov_mean × cd5,incremental,21392,0.2617
4,sem_nov_mean × cd10,disruptive-novel,22605,0.2748
5,sem_nov_mean × cd10,novel-consolidating,18633,0.2265
6,sem_nov_mean × cd10,conventional-disruptive,19094,0.2322
7,sem_nov_mean × cd10,incremental,21916,0.2665
8,sem_nov_min × cd5,disruptive-novel,23397,0.2863
9,sem_nov_min × cd5,novel-consolidating,17470,0.2137



mean×cd5 (primary) per-quadrant counts + shares:


,pairing,archetype,n,share
0,sem_nov_mean × cd5,disruptive-novel,22662,0.2773
1,sem_nov_mean × cd5,novel-consolidating,18210,0.2228
2,sem_nov_mean × cd5,conventional-disruptive,19474,0.2382
3,sem_nov_mean × cd5,incremental,21392,0.2617


## 5. Temporal trend (primary pairing mean×cd5)

Quadrant **shares by year** for the primary `sem_nov_mean×cd5` pairing, using the fixed thresholds already baked into `arch_mean_cd5`. We also report per-year **CD coverage** = fraction of that year's papers with a non-null `cd5`. Early years have sparse CD coverage (the OpenAlex forward-citation harvest is thin for old works), so early-year shares should not be over-read. Plots are inline-only — the **only** file written to `docs/figures` is `F2_dual_novelty.png`.

In [9]:
PRIMARY_COL = arch_col("sem_nov_mean", "cd5")
QORDER = list(QUADRANT_LABELS.values())

# Restrict to the modeled corpus span 1995-2025 (drop the lone 2026 stragglers).
yr = df[(df["year"] >= 1995) & (df["year"] <= 2025)].copy()

# Quadrant shares per year (over binned rows in that year).
binned = yr.dropna(subset=[PRIMARY_COL])
share_by_year = (
    binned.groupby("year")[PRIMARY_COL]
    .value_counts(normalize=True)
    .unstack()
    .reindex(columns=QORDER)
)
share_by_year = share_by_year.sort_index()
print("Quadrant shares by year (mean×cd5) — head & tail:")
with pd.option_context("display.float_format", lambda v: f"{v:.3f}"):
    display(pd.concat([share_by_year.head(4), share_by_year.tail(4)]))

# Per-year CD coverage.
cd_cov = yr.groupby("year").apply(
    lambda g: pd.Series({"n_papers": len(g), "cd5_nonnull": int(g["cd5"].notna().sum())}),
    include_groups=False,
)
cd_cov["cd5_coverage"] = cd_cov["cd5_nonnull"] / cd_cov["n_papers"]
print("\nPer-year CD coverage (fraction of papers with non-null cd5):")
with pd.option_context("display.float_format", lambda v: f"{v:.3f}"):
    display(cd_cov)

Quadrant shares by year (mean×cd5) — head & tail:


arch_mean_cd5,disruptive-novel,novel-consolidating,conventional-disruptive,incremental
year,,,,
1996,0.325,0.252,0.201,0.221
1997,0.318,0.257,0.205,0.221
1998,0.347,0.221,0.210,0.222
1999,0.316,0.231,0.214,0.239
2022,0.283,0.186,0.289,0.242
2023,0.300,0.183,0.289,0.228
2024,0.315,0.175,0.289,0.221
2025,0.282,0.185,0.301,0.232



Per-year CD coverage (fraction of papers with non-null cd5):


,n_papers,cd5_nonnull,cd5_coverage
year,,,
1995,2271,2127,0.937
1996,2342,2212,0.944
1997,2219,2119,0.955
1998,2242,2126,0.948
1999,2222,2115,0.952
2000,2232,2148,0.962
2001,2448,2330,0.952
2002,2522,2390,0.948
2003,2647,2535,0.958


In [10]:
# Inline-only stacked-area view of the 4 quadrant shares across years.
fig2, ax = plt.subplots(figsize=(11, 5))
sba = share_by_year.fillna(0.0)
ax.stackplot(
    sba.index.to_numpy(),
    [sba[q].to_numpy() for q in QORDER],
    labels=QORDER,
    alpha=0.85,
)
ax.set_xlabel("year")
ax.set_ylabel("quadrant share")
ax.set_ylim(0, 1)
ax.set_title("Quadrant shares by year (sem_nov_mean × cd5) — inline only, not saved")
ax.legend(loc="upper center", ncol=4, fontsize=8, framealpha=0.9)
fig2.tight_layout()
display(fig2)
plt.close(fig2)

<Figure size 1100x500 with 1 Axes>

## 6. By journal (primary pairing mean×cd5)

Quadrant shares per journal over binned `arch_mean_cd5` rows. Table; an inline bar plot only (not saved).

In [11]:
binned_all = df.dropna(subset=[PRIMARY_COL])
jshare = (
    binned_all.groupby("journal")[PRIMARY_COL]
    .value_counts(normalize=True)
    .unstack()
    .reindex(columns=QORDER)
)
jn = binned_all.groupby("journal").size().rename("n_binned")
jshare = jshare.join(jn).sort_values("n_binned", ascending=False)
print("Quadrant shares per journal (mean×cd5):")
with pd.option_context("display.float_format", lambda v: f"{v:.3f}"):
    display(jshare)

Quadrant shares per journal (mean×cd5):


,disruptive-novel,novel-consolidating,conventional-disruptive,incremental,n_binned
journal,,,,,
Spine,0.261,0.221,0.259,0.259,13623
The Journal of arthroplasty,0.207,0.178,0.318,0.297,11093
Clinical orthopaedics and related research,0.319,0.266,0.184,0.231,9933
Annals of surgery,0.297,0.222,0.230,0.250,8249
Arthroscopy : the journal of arthroscopic & related surgery : official publication of the Arthroscopy Association of North America and the International Arthroscopy Association,0.195,0.225,0.215,0.365,8215
Surgery,0.337,0.242,0.202,0.220,8090
The Journal of bone and joint surgery. American volume,0.270,0.225,0.243,0.262,7395
The British journal of surgery,0.267,0.234,0.219,0.280,5960
Journal of the American College of Surgeons,0.350,0.213,0.233,0.204,4713


In [12]:
# Inline-only grouped bar of journal quadrant shares.
fig3, ax = plt.subplots(figsize=(12, 5.5))
plot_j = jshare[QORDER]
plot_j.plot(kind="bar", ax=ax, width=0.8)
ax.set_ylabel("quadrant share")
ax.set_xlabel("journal")
ax.set_title("Quadrant shares by journal (sem_nov_mean × cd5) — inline only, not saved")
ax.legend(fontsize=8, ncol=4)
ax.tick_params(axis="x", rotation=45)
for lbl in ax.get_xticklabels():
    lbl.set_ha("right")
fig3.tight_layout()
display(fig3)
plt.close(fig3)

/tmp/claude-501/ipykernel_14124/1041658946.py:12: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  fig3.tight_layout()


<Figure size 1200x550 with 1 Axes>

## 7. Archetype × citation impact (age-fair)

Distribution of `cited_by_pctile_within_year` by archetype (mean×cd5). The **within-year percentile** is age-fair (a 1996 paper and a 2023 paper are each ranked against their own cohort), unlike raw `cited_by_count`. Per-archetype `describe()` + median percentile; inline boxplot only.

In [13]:
imp = df.dropna(subset=[PRIMARY_COL, "cited_by_pctile_within_year"])
imp_desc = imp.groupby(PRIMARY_COL)["cited_by_pctile_within_year"].describe().reindex(QORDER)
print("cited_by_pctile_within_year by archetype (mean×cd5):")
with pd.option_context("display.float_format", lambda v: f"{v:.4f}"):
    display(imp_desc)

med_pctile = imp.groupby(PRIMARY_COL)["cited_by_pctile_within_year"].median().reindex(QORDER)
print("\nPer-archetype MEDIAN within-year citation percentile:")
for q in QORDER:
    print(f"    {q:>26} : {med_pctile[q]:.4f}")

cited_by_pctile_within_year by archetype (mean×cd5):


,count,mean,std,min,25%,50%,75%,max
arch_mean_cd5,,,,,,,,
disruptive-novel,22662.0000,0.4582,0.2845,0.0019,0.2051,0.4329,0.6993,1.0000
novel-consolidating,18210.0000,0.4989,0.2784,0.0029,0.2621,0.5023,0.7376,1.0000
conventional-disruptive,19474.0000,0.5237,0.2816,0.0019,0.2820,0.5234,0.7692,1.0000
incremental,21392.0000,0.5749,0.2743,0.0034,0.3532,0.6069,0.8125,1.0000



Per-archetype MEDIAN within-year citation percentile:
              disruptive-novel : 0.4329
           novel-consolidating : 0.5023
       conventional-disruptive : 0.5234
                   incremental : 0.6069


In [14]:
# Inline-only boxplot of within-year citation percentile by archetype.
fig4, ax = plt.subplots(figsize=(9, 5))
data = [imp.loc[imp[PRIMARY_COL] == q, "cited_by_pctile_within_year"].to_numpy() for q in QORDER]
ax.boxplot(data, labels=QORDER, showfliers=False)
ax.set_ylabel("cited_by_pctile_within_year")
ax.set_title("Within-year citation percentile by archetype (mean×cd5) — inline only, not saved")
ax.tick_params(axis="x", rotation=20)
fig4.tight_layout()
display(fig4)
plt.close(fig4)

/tmp/claude-501/ipykernel_14124/1116288127.py:4: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=QORDER, showfliers=False)


<Figure size 900x500 with 1 Axes>

## 8. Candidate observations (for the gate)

Neutral, descriptive observations of the most notable patterns — temporal drift, journal contrasts, and impact-by-archetype — phrased as observations, NOT judgments. These feed the G3 report's "Candidate observations" section; whether any is "surprising" is Samer's call.

In [15]:
obs = []

# --- Temporal drift (mean×cd5) ---
early = share_by_year.loc[share_by_year.index <= 2000].mean()
late = share_by_year.loc[share_by_year.index >= 2020].mean()
drift = (late - early).sort_values()
biggest_up = drift.idxmax()
biggest_down = drift.idxmin()
early_cov = float(cd_cov.loc[cd_cov.index <= 2000, "cd5_coverage"].mean())
late_cov = float(cd_cov.loc[cd_cov.index >= 2020, "cd5_coverage"].mean())
obs.append(
    f"Temporal drift (mean×cd5): from <=2000 to >=2020, the '{biggest_up}' share moves "
    f"{drift[biggest_up]:+.3f} and '{biggest_down}' moves {drift[biggest_down]:+.3f}; "
    f"mean CD coverage rises from {early_cov:.2f} (<=2000) to {late_cov:.2f} (>=2020), "
    f"so early-year shares rest on a thin CD denominator and should not be over-read."
)

# --- Journal contrasts (mean×cd5) ---
dn = jshare["disruptive-novel"].dropna()
inc = jshare["incremental"].dropna()
obs.append(
    f"Journal contrasts (mean×cd5): 'disruptive-novel' share ranges "
    f"{dn.min():.3f} ({dn.idxmin()}) to {dn.max():.3f} ({dn.idxmax()}); "
    f"'incremental' share ranges {inc.min():.3f} ({inc.idxmin()}) to "
    f"{inc.max():.3f} ({inc.idxmax()}) across the {len(jshare)} journals."
)

# --- Impact by archetype (mean×cd5) ---
hi_imp = med_pctile.idxmax()
lo_imp = med_pctile.idxmin()
obs.append(
    f"Impact by archetype (mean×cd5, age-fair within-year percentile): median percentile "
    f"is highest for '{hi_imp}' ({med_pctile[hi_imp]:.3f}) and lowest for '{lo_imp}' "
    f"({med_pctile[lo_imp]:.3f}); full per-archetype medians: "
    + ", ".join(f"{q}={med_pctile[q]:.3f}" for q in QORDER)
    + "."
)

# --- Dual-novelty separation (the gate's headline) ---
obs.append(
    f"Dual-novelty separation: across all four pairings the largest |Pearson r| is "
    f"{max_abs_pearson:.3f} ({max_abs_pearson_pairing}) and the largest |Spearman ρ| is "
    f"{max_abs_spearman:.3f} ({max_abs_spearman_pairing}); both below 0.4, consistent with "
    f"semantic and structural novelty being weakly-correlated, distinct axes."
)

print("CANDIDATE OBSERVATIONS (neutral):\n")
for i, o in enumerate(obs, 1):
    print(f"  {i}. {o}\n")

CANDIDATE OBSERVATIONS (neutral):

  1. Temporal drift (mean×cd5): from <=2000 to >=2020, the 'conventional-disruptive' share moves +0.083 and 'novel-consolidating' moves -0.056; mean CD coverage rises from 0.95 (<=2000) to 0.90 (>=2020), so early-year shares rest on a thin CD denominator and should not be over-read.

  2. Journal contrasts (mean×cd5): 'disruptive-novel' share ranges 0.195 (Arthroscopy : the journal of arthroscopic & related surgery : official publication of the Arthroscopy Association of North America and the International Arthroscopy Association) to 0.396 (JAMA surgery); 'incremental' share ranges 0.155 (JAMA surgery) to 0.365 (Arthroscopy : the journal of arthroscopic & related surgery : official publication of the Arthroscopy Association of North America and the International Arthroscopy Association) across the 11 journals.

  3. Impact by archetype (mean×cd5, age-fair within-year percentile): median percentile is highest for 'incremental' (0.607) and lowest for 'd

## 9. Figure provenance

Record the F2 sidecar with the load-bearing gate numbers and figure settings.

In [16]:
sidecar_out = record_run(
    artifact_path=F2_FIG,
    inputs={"archetypes": ARCH_PATH},
    config={
        "figure": "F2_dual_novelty",
        "session": "V1-S11",
        "pairings": [f"{s} x {t}" for s, t in PAIRINGS],
        "n_complete_cases": int(len(COMPLETE)),
        "max_abs_pearson": max_abs_pearson,
        "max_abs_pearson_pairing": str(max_abs_pearson_pairing),
        "max_abs_spearman": max_abs_spearman,
        "max_abs_spearman_pairing": str(max_abs_spearman_pairing),
        "dpi": DPI,
        "hexbin_gridsize": GRIDSIZE,
        "thresholds": {k: float(v) for k, v in thresholds.items()},
        "source_config_hash": sidecar["config_hash"],
        "source_git_sha": sidecar["git_sha"],
    },
)
print("recorded run sidecar:", sidecar_out)
print("F2 PNG exists:", F2_FIG.exists(), "size KB:", f"{F2_FIG.stat().st_size / 1024:.1f}")

recorded run sidecar: /Users/samersalman/Desktop/SciField/docs/figures/F2_dual_novelty.png.run.json
F2 PNG exists: True size KB: 510.1
